In [ ]:
from cats_ai.evaluation import experiment

In [ ]:

if __name__ == "__main__":
    experiment()
    # query("CarCrash/videos/Normal/000007.mp4", ACCIDENT_PREDICTION_PROMPT, True)

In [ ]:
from cats_ai.prompts import ACCIDENT_ANALYSIS_PROMPT
from cats_ai.config import MAX_NEW_TOKENS, NFRAMES
import cv2
import base64
from openai import OpenAI
import os


def capture_frames(video_path: str, n_frames: int):
    cap = cv2.VideoCapture(video_path)
    base64Frames = []

    for i in range(n_frames):
        ret, frame = cap.read()
        if not ret:
            break

        _, buffer = cv2.imencode(".jpg", frame)
        encoded_image = base64.b64encode(buffer).decode("utf-8")
        base64Frames.append(encoded_image)

    cap.release()
    return base64Frames


def chatgpt_query(video_path: str, prompt: str, n_frames: int) -> str:
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    base64_frames = capture_frames(video_path, n_frames)

    content = [
        {"type": "input_text", "text": prompt},
        *[
            {
                "type": "input_image",
                "image_url": f"data:image/jpeg;base64,{frame}",
            }
            for frame in base64_frames
        ],
    ]

    response = client.responses.create(
        model="gpt-5.5",
        input=[{"role": "user", "content": content}],
        max_output_tokens=MAX_NEW_TOKENS,
    )

    return response.output_text


chatgpt_query(
    "CarCrash/videos/Crash-1500/000001.mp4", ACCIDENT_ANALYSIS_PROMPT, NFRAMES
)
